# Week 4 — Compress, Simulate, Predict

## Overview

**Research question:** Can we choose a small set of contracts that supports accurate valuation of many other contracts?

This week connects unsupervised learning to an actuarial pricing task. We compare random representatives with actual contracts selected from K-means clusters, simulate the representatives, and copy the nearest representative's normalized liability. We measure both geometric coverage and valuation error: they answer different questions.

**Learning objectives**

- Explain the compressor–simulator–predictor framework in your own words.
- Select distinct representatives using candidate features only.
- Keep the simulation budget comparable when changing the compressor.
- Distinguish coverage, contract valuation error, and aggregate valuation error.
- Account for reference simulation, validation simulation, and computation.

**Prerequisites:** Weeks 1–3; arrays, means, Euclidean distance, and Monte Carlo standard errors. The notebook contains its own helper functions and does not require files from earlier weeks.

**Output contract:** one two-method comparison table, two diagnostic figures, one explicit cost ledger, and a short interpretation supported by your observed results. No model ranking is supplied in advance.

**Suggested workload:** 75-minute meeting plus 4–6 independent hours, including reading and homework. Complete the guided investigation first; the final exercise is optional.

**Study boundary:** this is a fixed-fee, survival-conditioned GMAB top-up problem. The held-out test split remains closed through Week 7; Week 8 is the final assessment.

## 1. A valuation pipeline with three decisions

A compressor chooses a small reference set; a simulator supplies expensive labels; a predictor transfers information from those labels to other contracts.

| Component | Input | Output | Decision held fixed or changed this week |
|---|---|---|---|
| Compressor | Unlabelled candidate features | Indices of actual reference contracts | Compare random selection and K-means member selection |
| Simulator | Reference contracts and a path budget | MC means and standard errors | Same paths per reference for both compressors |
| Predictor | Reference features and MC means | Predicted normalized liabilities | Same Euclidean nearest-reference copying rule |

For contract features \(x=(a,g,T,\sigma,\phi)\), our target is

\[
y(x)=E_Q\!\left[e^{-rT}\left(g-ae^{(r-\phi-\sigma^2/2)T+\sigma\sqrt{T}Z}\right)_+\right],
\qquad Z\sim N(0,1).
\]

The currency liability is \(L=P\,y(x)\). Here \(a=A_0/P\), \(g=G/P\), and \(\phi\) is a specified continuous fee rate. This is the insurer's **gross terminal top-up liability**: fee income is not subtracted. In this simplified model, a higher fee reduces the account and can increase this top-up. It is not a fair-fee equation or the full policyholder value.

**Counting example.** Valuing 1,000 contracts at 2,048 paths each uses 2,048,000 contract-path payoffs. Valuing 40 references uses 81,920. Simulating 100 validation contracts adds another 204,800. These are counts, not measured runtime speedups. A cheap teaching payoff may make model overhead relatively important.

The broad three-component framework motivates this exercise; the implementation below is an undergraduate adaptation. It does not reproduce the SRSC compressor, mixed-data k-prototypes, or optimal simulation allocation in Feng, Tan, and Zheng (2020).

In [ ]:
# Complete common pricing code. No data files or earlier notebooks are required.
import sys
import json
import platform
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects:
            print(obj)

RISK_FREE_RATE = 0.03
FEATURES = ['account_ratio', 'guarantee_ratio', 'maturity', 'volatility', 'fee']
plt.rcParams.update({'figure.figsize': (7.2, 4.3), 'axes.grid': True,
                     'grid.alpha': 0.22, 'font.size': 10})


def make_portfolio(n=1200, seed=2027):
    """Synthetic in-force GMAB contracts, never customer data.

    P is a reference premium, A0=P*account_ratio, and G=P*guarantee_ratio.
    T is the remaining term. All contracts share the same risk-free rate.
    Volatility describes the contract's fund; rates are annual decimals.
    """
    if not isinstance(n, (int, np.integer)) or n < 10:
        raise ValueError('n must be an integer of at least 10.')
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'contract_id': np.arange(n, dtype=int),
        'premium': rng.uniform(10_000.0, 50_000.0, n),
        'account_ratio': rng.uniform(0.65, 1.40, n),
        'guarantee_ratio': rng.uniform(0.90, 1.60, n),
        'maturity': rng.integers(5, 21, n).astype(float),
        'volatility': rng.uniform(0.12, 0.35, n),
        'fee': rng.uniform(0.005, 0.030, n),
    })


def validate_contracts(df):
    required = ['contract_id', 'premium'] + FEATURES
    missing = [c for c in required if c not in df]
    if missing:
        raise ValueError(f'Missing contract columns: {missing}')
    if len(df) == 0:
        raise ValueError('Supply at least one contract.')
    values = df[required].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('All contract fields must be finite.')
    ids = df['contract_id'].to_numpy(dtype=float)
    if (ids < 0).any() or (ids != np.floor(ids)).any() or len(np.unique(ids)) != len(ids):
        raise ValueError('contract_id must contain unique nonnegative integers.')
    if (df[['premium', 'account_ratio', 'guarantee_ratio']] <= 0).any().any():
        raise ValueError('Premium and account/guarantee ratios must be positive.')
    if (df[['maturity', 'volatility', 'fee']] < 0).any().any():
        raise ValueError('Maturity, volatility and fee must be nonnegative.')


def gmab_mc(df, n_paths=2048, seed=123):
    """Independent plain Monte Carlo for per-premium gross top-up liability.

    Identical (seed, contract_id) reproduces the same path stream even when
    the contract appears in a different subset or row order. Distinct IDs
    give independent streams. This is NOT a common-market portfolio-risk
    simulator: it estimates a sum of expected liabilities, not portfolio VaR.
    SE describes Monte Carlo sampling uncertainty only.
    """
    validate_contracts(df)
    if not isinstance(n_paths, (int, np.integer)) or n_paths < 2:
        raise ValueError('n_paths must be an integer of at least 2.')
    if not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('seed must be a nonnegative integer.')
    values, errors, sds = [], [], []
    for row in df.itertuples():
        rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(row.contract_id)]))
        z = rng.standard_normal(n_paths)
        terminal_ratio = row.account_ratio * np.exp(
            (RISK_FREE_RATE - row.fee - 0.5 * row.volatility**2) * row.maturity
            + row.volatility * np.sqrt(row.maturity) * z
        )
        payoff = np.exp(-RISK_FREE_RATE * row.maturity) * np.maximum(
            row.guarantee_ratio - terminal_ratio, 0.0)
        sd = float(np.std(payoff, ddof=1))
        values.append(float(np.mean(payoff)))
        errors.append(sd / np.sqrt(n_paths))
        sds.append(sd)
    return pd.DataFrame({'value': values, 'se': errors, 'payoff_sd': sds,
                         'paths': int(n_paths)}, index=df.index)


def gmab_exact(df):
    """Closed-form put benchmark under the simplified model: audit only.

    Never use these values as training or tuning labels in the project.
    The deterministic cases T=0 or sigma=0 are handled separately.
    """
    validate_contracts(df)
    a = df['account_ratio'].to_numpy(dtype=float)
    g = df['guarantee_ratio'].to_numpy(dtype=float)
    t = df['maturity'].to_numpy(dtype=float)
    sig = df['volatility'].to_numpy(dtype=float)
    fee = df['fee'].to_numpy(dtype=float)
    ans = np.maximum(g * np.exp(-RISK_FREE_RATE*t) - a * np.exp(-fee*t), 0.0)
    regular = (t > 0) & (sig > 0)
    if np.any(regular):
        tr, sr = t[regular], sig[regular]
        d1 = (np.log(a[regular] / g[regular]) +
              (RISK_FREE_RATE-fee[regular]+0.5*sr**2)*tr) / (sr*np.sqrt(tr))
        d2 = d1 - sr*np.sqrt(tr)
        ans[regular] = (g[regular]*np.exp(-RISK_FREE_RATE*tr)*norm.cdf(-d2)
                        - a[regular]*np.exp(-fee[regular]*tr)*norm.cdf(-d1))
    return np.maximum(ans, 0.0)


def split_portfolio(df, seed=2027):
    """Split contracts before building pairs, selecting labels or tuning."""
    validate_contracts(df)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    n_candidate, n_validation = int(0.60*len(df)), int(0.20*len(df))
    if min(n_candidate, n_validation, len(df)-n_candidate-n_validation) < 1:
        raise ValueError('Portfolio is too small for the requested split.')
    return {
        'candidate': df.iloc[order[:n_candidate]].copy(),
        'validation': df.iloc[order[n_candidate:n_candidate+n_validation]].copy(),
        'test': df.iloc[order[n_candidate+n_validation:]].copy(),
    }


def score_predictions(y_true, y_pred, premium=None):
    """Errors per premium, plus signed aggregate relative error when requested."""
    truth = np.asarray(y_true, dtype=float).reshape(-1)
    pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if truth.shape != pred.shape or len(truth) == 0:
        raise ValueError('Truth and predictions must have equal nonzero length.')
    if not (np.isfinite(truth).all() and np.isfinite(pred).all()):
        raise ValueError('Truth and predictions must be finite.')
    error = pred-truth
    out = {'rmse': float(np.sqrt(np.mean(error**2))),
           'mae': float(np.mean(np.abs(error))),
           'p95_abs': float(np.quantile(np.abs(error), 0.95)),
           'bias': float(np.mean(error)),
           'negative_fraction': float(np.mean(pred < 0))}
    if premium is not None:
        p = np.asarray(premium, dtype=float).reshape(-1)
        if p.shape != truth.shape or not np.isfinite(p).all() or np.any(p <= 0):
            raise ValueError('Premium weights must be finite, positive and aligned.')
        denominator = float(p @ truth)
        out['portfolio_relative_error'] = (float(p @ error)/denominator
                                            if abs(denominator) > 1e-12 else np.nan)
    return out


In [ ]:
# Complete shared learning code. All labels are supplied explicitly by the caller.
from scipy.spatial.distance import cdist
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler, SplineTransformer
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline


def build_features(candidate, query=None, columns=FEATURES):
    """Fit feature scales only on the unlabelled candidate pool.

    A constant training column maps to zero on training data. A held-out
    observation outside the training range may map outside [0,1]; no clipping.
    No premium, contract ID or computed valuation is an informative feature.
    """
    scaler = MinMaxScaler()
    z_candidate = scaler.fit_transform(candidate[list(columns)])
    z_query = None if query is None else scaler.transform(query[list(columns)])
    return scaler, z_candidate, z_query


def select_representatives(Z, m, method='kmeans', seed=2027):
    """Return m DISTINCT row positions of actual candidate contracts.

    K-means is fitted to candidate FEATURES only. For each nonempty cluster,
    choose the member closest to that cluster's center. This is a member
    representative, not an exact k-medoids optimization or Feng et al.'s SRSC.
    """
    z = np.asarray(Z, dtype=float)
    if z.ndim != 2 or not np.isfinite(z).all():
        raise ValueError('Z must be a finite 2D matrix.')
    if not isinstance(m, (int, np.integer)) or not 1 <= m <= len(z):
        raise ValueError('m must lie between 1 and the number of candidates.')
    rng = np.random.default_rng(seed)
    if method == 'random':
        return np.sort(rng.choice(len(z), size=m, replace=False))
    if method != 'kmeans':
        raise ValueError("method must be 'random' or 'kmeans'.")
    km = KMeans(n_clusters=m, n_init=5, max_iter=150, random_state=seed)
    membership = km.fit_predict(z)
    chosen = []
    for cluster in range(m):
        members = np.flatnonzero(membership == cluster)
        if len(members):
            distances = np.sum((z[members] - km.cluster_centers_[cluster])**2, axis=1)
            chosen.append(int(members[np.argmin(distances)]))
    # Duplicate feature vectors can yield fewer than m nonempty clusters.
    # Keep the requested contract count without ever repeating an ID.
    if len(chosen) < m:
        remaining = np.setdiff1d(np.arange(len(z)), chosen)
        chosen.extend(rng.choice(remaining, m-len(chosen), replace=False).tolist())
    return np.sort(np.asarray(chosen, dtype=int))


def nearest_predict(Zref, yref, Zquery, weights=None, k=1):
    """Uniform k-neighbor transfer; k=1 is the CSM-style teaching baseline.

    A diagonal weighted distance is ordinary Euclidean distance after each
    coordinate is multiplied by sqrt(weight). Stable sort breaks exact ties
    in reference-row order. Chunking avoids an N-by-N distance matrix.
    """
    zr, zq = np.asarray(Zref, dtype=float), np.asarray(Zquery, dtype=float)
    yr = np.asarray(yref, dtype=float).reshape(-1)
    if zr.ndim != 2 or zq.ndim != 2 or zr.shape[1] != zq.shape[1] or len(zr) != len(yr):
        raise ValueError('Reference/query dimensions or labels do not align.')
    if not (np.isfinite(zr).all() and np.isfinite(zq).all() and np.isfinite(yr).all()):
        raise ValueError('Features and labels must be finite.')
    if not isinstance(k, (int, np.integer)) or not 1 <= k <= len(zr):
        raise ValueError('k must be between 1 and the reference count.')
    w = np.ones(zr.shape[1]) if weights is None else np.asarray(weights, dtype=float)
    if w.shape != (zr.shape[1],) or not np.isfinite(w).all() or np.any(w < 0):
        raise ValueError('Distance weights must be finite and nonnegative.')
    scale = np.sqrt(w)
    neighbors = np.empty((len(zq), k), dtype=int)
    for start in range(0, len(zq), 1024):
        stop = min(start+1024, len(zq))
        distance = cdist(zq[start:stop]*scale, zr*scale, metric='sqeuclidean')
        neighbors[start:stop] = np.argsort(distance, axis=1, kind='stable')[:, :k]
    return yr[neighbors].mean(axis=1), neighbors


def fit_metric(Zref, yref, lam=0.01, max_pairs=2000, seed=2027):
    """Fit a simple diagonal response-aware pseudometric on reference pairs.

    Objective: mean(((H@w)/p - ((y_i-y_j)/s)^2)^2)
               + lam/2 * sum((w-1)^2),  w>=0, sum(w)=p.
    Here H=(z_i-z_j)^2 coordinatewise and s=max(ptp(yref),1e-6).
    The label scale is part of the modeling choice, fitted on references only.
    Numerical labels have noise: this loss is not an unbiased exact-gap loss.
    """
    z, y = np.asarray(Zref, dtype=float), np.asarray(yref, dtype=float).reshape(-1)
    if z.ndim != 2 or len(z) != len(y) or len(y) < 2:
        raise ValueError('Supply at least two aligned labeled reference contracts.')
    if not (np.isfinite(z).all() and np.isfinite(y).all()):
        raise ValueError('Training data must be finite.')
    if lam < 0 or not isinstance(max_pairs, (int, np.integer)) or max_pairs < 1:
        raise ValueError('lam must be nonnegative and max_pairs a positive integer.')
    n, p = z.shape
    count = min(max_pairs, n*(n-1)//2)
    rng = np.random.default_rng(seed)
    if count == n*(n-1)//2:
        i, j = np.triu_indices(n, k=1)
    else:
        # Uniform directed non-self pairs sampled WITH replacement.
        # They share contracts and are not independent observational units.
        i = rng.integers(0, n, size=count)
        j = (i+rng.integers(1, n, size=count)) % n
    label_scale = max(float(np.ptp(y)), 1e-6)
    h = (z[i]-z[j])**2 / p
    target = ((y[i]-y[j])/label_scale)**2
    def objective(w):
        residual = h @ w-target
        return float(np.mean(residual**2) + 0.5*lam*np.sum((w-1.0)**2))
    def gradient(w):
        return 2.0*h.T @ (h @ w-target)/len(target) + lam*(w-1.0)
    result = minimize(objective, np.ones(p), jac=gradient, method='SLSQP',
                      bounds=[(0.0, float(p))]*p,
                      constraints=[{'type': 'eq', 'fun': lambda w: w.sum()-p,
                                    'jac': lambda w: np.ones(p)}],
                      options={'maxiter': 400, 'ftol': 1e-10})
    feasible = (np.isfinite(result.x).all() and result.x.min() >= -1e-7
                and abs(result.x.sum()-p) <= 1e-6)
    success = bool(result.success and feasible)
    weights = np.clip(result.x, 0.0, None) if success else np.ones(p)
    weights *= p/weights.sum()
    return {'weights': weights, 'success': success, 'pair_count': len(target),
            'objective': objective(weights), 'scale': label_scale,
            'iterations': int(result.nit), 'message': str(result.message)}


def make_regressors(seed=2027):
    """Fixed small model menu. Regression predicts values, not distances."""
    return {
        'Ridge': Ridge(alpha=1.0),
        'SplineGAM': make_pipeline(
            SplineTransformer(n_knots=4, degree=3, include_bias=False),
            Ridge(alpha=1.0)),
        'RandomForest': RandomForestRegressor(n_estimators=100, min_samples_leaf=3,
                                               random_state=seed, n_jobs=1),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=2,
                                                       learning_rate=0.05, random_state=seed),
    }


In [ ]:
# One fixed design for this notebook and the next two weeks.
PORTFOLIO_SEED = 2027
SPLIT_SEED = 2027
COMPRESSOR_SEED = 2027
REFERENCE_MC_SEED = 4101
VALIDATION_MC_SEED = 4102
N_PORTFOLIO = 1000
N_REFERENCES = 40
N_VALIDATION = 100
REFERENCE_PATHS = 2048
VALIDATION_PATHS = 2048

portfolio = make_portfolio(n=N_PORTFOLIO, seed=PORTFOLIO_SEED)
parts = split_portfolio(portfolio, seed=SPLIT_SEED)
candidate = parts["candidate"].copy()
validation = parts["validation"].sample(
    n=min(N_VALIDATION, len(parts["validation"])), random_state=2028
).copy()
# Do not access parts["test"], compute its labels, or inspect its predictions.

t0 = perf_counter()
scaler, Zcandidate, Zvalidation = build_features(candidate, validation)
feature_seconds = perf_counter() - t0

t0 = perf_counter()
validation_mc = gmab_mc(
    validation, n_paths=VALIDATION_PATHS, seed=VALIDATION_MC_SEED
)
validation_mc_seconds = perf_counter() - t0
y_validation = validation_mc["value"].to_numpy()
premium_validation = validation["premium"].to_numpy()

display(pd.DataFrame({
    "quantity": ["candidate contracts", "validation contracts used",
                 "reference count per compressor", "paths per reference",
                 "paths per validation contract"],
    "value": [len(candidate), len(validation), N_REFERENCES,
              REFERENCE_PATHS, VALIDATION_PATHS]
}))
print("No exact-price labels have been used. The test split has not been opened.")

## 2. What does an unsupervised compressor optimize?

Use the candidate-fitted transformation

\[
z_j(x)=\frac{x_j-\min_{\text{candidate}}x_j}
{\max_{\text{candidate}}x_j-\min_{\text{candidate}}x_j}.
\]

The implementation uses MinMaxScaler, including its handling of constant columns. A held-out feature can lie outside \([0,1]\); we do not refit the scaler to hide that fact. Premium and contract ID are excluded from the normalized pricing features.

K-means seeks centers \(c_1,\ldots,c_m\) and assignments that reduce

\[
\sum_{i\in\text{candidate}}\min_{1\leq j\leq m}\|z_i-c_j\|_2^2.
\]

A center averages features and need not be an actual contract. Our teaching compressor therefore selects the member closest to its own cluster center. It returns distinct candidate-row positions and fills missing selections if degenerate clusters occur. This is **not** an exact K-medoids optimization.

A different operation happens later: prediction assigns each query to its nearest selected reference. After replacing a center with an actual member, these prediction groups need not equal the original K-means memberships.

For query \(q\), define coverage radius

\[
\rho(q)=\min_{j\in\mathcal R}\|z(q)-z_j\|_2.
\]

Small radii mean the references cover feature space well. They do not by themselves establish that the references have similar liabilities: a feature can vary substantially while having little pricing effect, or vary slightly in a region with a steep pricing response.

**Worked example.** Suppose two candidate differences are \((0.01,0.20)\) and \((0.15,0)\) in two scaled features. Euclidean distance prefers the second, because \(0.15<\sqrt{0.01^2+0.20^2}\). Whether that is a better valuation match depends on how the target responds to both features. Week 6 will use labels to investigate that response.

In [ ]:
def coverage_radii(Zquery, Zref):
    """Distance to the nearest actual reference in candidate-scaled space."""
    squared = np.sum(
        (np.asarray(Zquery)[:, None, :] - np.asarray(Zref)[None, :, :])**2,
        axis=2
    )
    return np.sqrt(squared.min(axis=1))


def run_compressor_experiment(method, m=N_REFERENCES, seed=COMPRESSOR_SEED):
    """One educational same-path comparison; validation labels are reused."""
    start = perf_counter()

    t = perf_counter()
    positions = select_representatives(Zcandidate, m=m, method=method, seed=seed)
    compression_seconds = perf_counter() - t
    refs = candidate.iloc[positions].copy()
    Zref = Zcandidate[positions]

    t = perf_counter()
    reference_mc = gmab_mc(
        refs, n_paths=REFERENCE_PATHS, seed=REFERENCE_MC_SEED
    )
    reference_mc_seconds = perf_counter() - t
    yref = reference_mc["value"].to_numpy()

    t = perf_counter()
    prediction, neighbors = nearest_predict(Zref, yref, Zvalidation, k=1)
    prediction_seconds = perf_counter() - t

    # These are diagnostic computations, separate from deployed prediction.
    t = perf_counter()
    radii_candidate = coverage_radii(Zcandidate, Zref)
    radii_validation = coverage_radii(Zvalidation, Zref)
    metrics = score_predictions(
        y_validation, prediction, premium=premium_validation
    )
    diagnostic_seconds = perf_counter() - t
    run_seconds = perf_counter() - start

    row = {
        "compressor": method,
        "references": len(refs),
        "candidate_mean_radius": radii_candidate.mean(),
        "validation_mean_radius": radii_validation.mean(),
        "validation_p95_radius": np.quantile(radii_validation, 0.95),
        "validation_RMSE_pp": 100 * metrics["rmse"],
        "validation_MAE_pp": 100 * metrics["mae"],
        "validation_p95_abs_pp": 100 * metrics["p95_abs"],
        "validation_bias_pp": 100 * metrics["bias"],
        "validation_aggregate_error_pct": 100 * metrics["portfolio_relative_error"],
        "reference_payoff_paths": len(refs) * REFERENCE_PATHS,
        "compression_seconds": compression_seconds,
        "reference_MC_seconds": reference_mc_seconds,
        "prediction_seconds": prediction_seconds,
        "diagnostic_seconds": diagnostic_seconds,
        "standalone_development_seconds": (
            feature_seconds + validation_mc_seconds + run_seconds
        ),
    }
    return {
        "row": row, "positions": positions, "refs": refs, "Zref": Zref,
        "reference_mc": reference_mc, "prediction": prediction,
        "neighbors": neighbors, "validation_radii": radii_validation
    }


runs = {
    method: run_compressor_experiment(method)
    for method in ["random", "kmeans"]
}
comparison = pd.DataFrame([runs[method]["row"] for method in runs])
display(comparison[[
    "compressor", "references", "candidate_mean_radius",
    "validation_mean_radius", "validation_p95_radius",
    "validation_RMSE_pp", "validation_MAE_pp", "validation_p95_abs_pp",
    "validation_bias_pp", "validation_aggregate_error_pct"
]].round(5))

### Inspect the geometry without mistaking a projection for the full space

The next plot displays two of the five scaled features. All five features were used by the compressor. A reference that appears redundant in this plot may differ in account ratio, guarantee ratio, or fee.

Random selection uses the same reference count as K-means. K-means uses no MC labels, including no validation responses. Its access to the unlabelled candidate feature table is part of its computation and must be counted when measuring time.

In [ ]:
j_maturity = FEATURES.index("maturity")
j_volatility = FEATURES.index("volatility")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True, sharey=True)
for ax, method in zip(axes, ["random", "kmeans"]):
    ax.scatter(Zcandidate[:, j_maturity], Zcandidate[:, j_volatility],
               s=13, alpha=0.18, color="#55758c", label="Candidate")
    selected = runs[method]["Zref"]
    ax.scatter(selected[:, j_maturity], selected[:, j_volatility],
               s=48, facecolors="none", edgecolors="#be5a38",
               linewidths=1.5, label="Selected reference")
    ax.set(title=method.capitalize() + " selection",
           xlabel="Scaled maturity", ylabel="Scaled volatility")
    ax.legend()
fig.suptitle("Two-feature projection; selection uses all five features")
fig.tight_layout()
plt.show()

## 3. The simulator supplies noisy responses

For reference \(j\), simulate \(b\) independent payoffs \(Y_{j1},\ldots,Y_{jb}\):

\[
\widehat y_j=\frac1b\sum_{s=1}^{b}Y_{js},
\qquad
\widehat{\mathrm{se}}_j
=\frac{\mathrm{sd}(Y_{j1},\ldots,Y_{jb};\ \mathrm{ddof}=1)}{\sqrt b}.
\]

A response is a Monte Carlo estimate, not an error-free observation. Using the same number of paths does not produce the same standard error for every contract, because payoff variances differ.

The helper uses a seed determined by the experiment seed and contract ID. Thus a reference appearing in both compressors receives the same label, while different contract IDs have independent streams. The validation experiment uses a different seed and different contracts.

**A budget trade-off.** With a fixed reference-path budget \(B_{\mathrm{ref}}=mb\), increasing \(m\) may improve coverage but reduces paths per reference if the total is held fixed. That makes the individual responses noisier. This week keeps \(m\) and \(b\) equal across the two compressors to isolate the compressor comparison. Homework explores \(m\); it must disclose the resulting budget change.

The final benchmark in Weeks 7–8 will use a declared total development budget, including validation labels. Do not present today's same-path experiment as proof of optimal allocation or as an equal-total-budget comparison with MC-all.

In [ ]:
uncertainty_table = pd.DataFrame([
    {
        "compressor": method,
        "median_reference_SE_pp": 100 * run["reference_mc"]["se"].median(),
        "max_reference_SE_pp": 100 * run["reference_mc"]["se"].max(),
        "reference_contract_valuations": len(run["refs"]),
        "reference_payoff_paths": len(run["refs"]) * REFERENCE_PATHS,
    }
    for method, run in runs.items()
])
display(uncertainty_table.round(5))
print("Shared validation contract valuations:", len(validation))
print("Shared validation payoff paths:", len(validation) * VALIDATION_PATHS)
print("Validation RMS standard error, in percentage points of premium:",
      round(100 * np.sqrt(np.mean(validation_mc["se"].to_numpy()**2)), 5))

## 4. Prediction by copying the nearest reference

For a new contract \(q\), the baseline selects

\[
j^*(q)=\arg\min_{j\in\mathcal R}\|z(q)-z_j\|_2,
\qquad
\widehat y(q)=\widehat y_{j^*(q)}.
\]

The currency prediction is always

\[
\widehat L(q)=P_q\,\widehat y(q).
\]

**Worked example.** If the copied normalized liability is \(0.04\), a query with premium 10,000 has a predicted top-up liability of 400; a query with premium 30,000 has a predicted liability of 1,200. Copying the reference's currency liability would be wrong when premiums differ.

Feng, Tan, and Zheng call their predictor **cluster size multiple (CSM)**. Section 5, Equation (12), transfers the nearest prototype's simulated value. Our per-premium copying rule is CSM-style; our compressor and simplified target differ. The paper's nearest-neighbor **cleanup** is a separate compressor operation discussed in Section 3.1, not the name of its predictor.

For a group assigned to reference \(j\), the currency aggregate here is
\(\widehat y_j\sum_{q:\,j^*(q)=j}P_q\).
Multiplying by only the number of group members would instead total normalized values, unless all premiums were equal and the common premium were also included.

Nearest-reference copying is piecewise constant. It can be useful as a transparent benchmark even when a smoother predictor is ultimately more accurate. That is an empirical question.

In [ ]:
# A small currency-unit example independent of the synthetic results.
currency_example = pd.DataFrame({
    "query_premium": [10_000.0, 30_000.0],
    "copied_normalized_liability": [0.04, 0.04]
})
currency_example["predicted_currency_liability"] = (
    currency_example["query_premium"]
    * currency_example["copied_normalized_liability"]
)
display(currency_example)

# Check the portfolio calculation using the actual validation predictions.
transfer_totals = []
for method, run in runs.items():
    selected_neighbor = run["neighbors"][:, 0]
    group_premiums = np.bincount(
        selected_neighbor, weights=premium_validation, minlength=len(run["refs"])
    )
    copied_labels = run["reference_mc"]["value"].to_numpy()
    direct_sum = float(premium_validation @ run["prediction"])
    grouped_sum = float(group_premiums @ copied_labels)
    transfer_totals.append({
        "compressor": method,
        "sum_of_currency_predictions": direct_sum,
        "premium_weighted_group_sum": grouped_sum,
        "absolute_difference": abs(direct_sum - grouped_sum)
    })
display(pd.DataFrame(transfer_totals).round(6))

## 5. Evaluate the pricing task and the cost honestly

We evaluate predictions against independently simulated **validation MC means**:

\[
\mathrm{RMSE}_{\mathrm{val}}
=\sqrt{\frac1v\sum_{q=1}^{v}
  \left(\widehat y(q)-\widehat y^{\,\mathrm{MC}}_{\mathrm{val}}(q)\right)^2}.
\]

The target in this score is noisy. For a fixed predictor that is independent of validation simulation noise, expected observed squared error equals true squared prediction error plus validation MC variance. This does not make a selected best validation score an unbiased final accuracy estimate: selecting on the same validation results introduces optimism.

Report MAE, a large-error quantile, and signed bias as well as RMSE. Multiply a normalized error by 100 to express **percentage points of premium**. For example, an error of 0.01 is one percentage point of premium, not a one-percent relative error in the liability.

The aggregate error uses premium weights:

\[
\frac{\sum_q P_q(\widehat y(q)-\widehat y^{\,\mathrm{MC}}_{\mathrm{val}}(q))}
     {\sum_q P_q\widehat y^{\,\mathrm{MC}}_{\mathrm{val}}(q)}.
\]

Positive and negative contract errors may cancel, so a small aggregate error does not establish accurate individual valuations. The denominator can also be small for other portfolios.

**Interpretation prompts**

1. Does the better geometric coverage in your run correspond to lower valuation RMSE? Cite both numbers.
2. Are the largest valuation errors necessarily the largest coverage radii?
3. How large is validation MC uncertainty compared with the difference between methods?
4. What would another random seed test? What would another market model test?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
colors = {"random": "#397a9c", "kmeans": "#bf613b"}
for method, run in runs.items():
    error_pp = 100 * (run["prediction"] - y_validation)
    axes[0].scatter(run["validation_radii"], np.abs(error_pp),
                    s=24, alpha=0.65, color=colors[method], label=method)
    axes[1].scatter(100 * y_validation, 100 * run["prediction"],
                    s=24, alpha=0.65, color=colors[method], label=method)

axes[0].set(xlabel="Distance to nearest reference (scaled features)",
            ylabel="Absolute error vs validation MC (pp of premium)",
            title="Coverage and valuation error")
all_values = np.concatenate(
    [100 * y_validation] + [100 * run["prediction"] for run in runs.values()]
)
lo, hi = float(all_values.min()), float(all_values.max())
axes[1].plot([lo, hi], [lo, hi], "--", color="#555555", linewidth=1)
axes[1].set(xlabel="Validation MC mean (% of premium)",
            ylabel="Predicted liability (% of premium)",
            title="Prediction against a noisy validation estimate")
for ax in axes:
    ax.legend()
fig.tight_layout()
plt.show()

time_columns = [
    "compressor", "compression_seconds", "reference_MC_seconds",
    "prediction_seconds", "diagnostic_seconds", "standalone_development_seconds"
]
display(comparison[time_columns].round(5))

campaign_ledger = pd.DataFrame([
    {"operation": "Reference simulation, random",
     "contract_valuations": N_REFERENCES,
     "payoff_paths": N_REFERENCES * REFERENCE_PATHS},
    {"operation": "Reference simulation, K-means",
     "contract_valuations": N_REFERENCES,
     "payoff_paths": N_REFERENCES * REFERENCE_PATHS},
    {"operation": "Validation simulation, shared once",
     "contract_valuations": len(validation),
     "payoff_paths": len(validation) * VALIDATION_PATHS}
])
display(campaign_ledger)
print("Actual guided campaign payoff paths:",
      int(campaign_ledger["payoff_paths"].sum()))
print("Feature preprocessing seconds:", round(feature_seconds, 5))
print("Shared validation simulation seconds:", round(validation_mc_seconds, 5))

### How to read the ledger

A contract valuation means one simulated contract response, even if a single Python function call returns many responses. A payoff-path count measures simulation work more directly than the number of function calls.

Each standalone development time includes feature preparation, its compressor, reference MC, prediction, diagnostics, and the shared validation MC cost. To report the cost of running both methods in this notebook, count the validation labels once rather than adding the two standalone totals. Repeated reference labels across the two methods are recomputed by this teaching code, so their executed work is counted twice.

Short wall-clock timings vary with the computer and background work. Repeat timings in the final benchmark. Fewer simulated paths can be valuable without guaranteeing a measured speedup in this inexpensive terminal-payoff example.

**Complete a research-log entry:** state which component changed, which quantities stayed fixed, the observed coverage and valuation results, the charged simulation counts, and one reason the result may change under another design.

## Reading assignment — approximately 45–60 minutes

1. **Ben Mingbin Feng, Zhenni Tan, and Jiayi Zheng (2020), _Efficient Simulation Designs for Valuation of Large Variable Annuity Portfolios_, North American Actuarial Journal 24(2), 275–289.** [Author PDF](https://www.math.uwaterloo.ca/~mbfeng/papers/2020_VAPort.pdf); [DOI](https://doi.org/10.1080/10920277.2019.1685394). Read Sections 3 and 5 selectively; Section 4 is optional this week.
2. **Guojun Gan and X. Sheldon Lin (2015), _Valuation of large variable annuity portfolios under nested simulation: A functional data approach_, Insurance: Mathematics and Economics 62, 138–150.** [DOI](https://doi.org/10.1016/j.insmatheco.2015.02.007). Read the abstract, introduction, and framework overview. Detailed kriging and nested-simulation derivations are not required.

**Submit brief answers**

- What are the three components, and which component did this notebook change?
- In Feng et al., how does SRSC differ from clustering the entire candidate table here?
- What does the CSM predictor copy? Why is the nearest-neighbor cleanup a different operation?
- What extra source of computational cost does nested simulation introduce beyond our terminal GMAB example?

## Homework 1 — More representatives, more work?

**Core; about 45–60 minutes.** Extend the starter to reference counts 20, 40, and 60 for both compressors. Retain the same validation labels and the same paths per reference. Plot validation RMSE against reference payoff paths and against reference count. Report actual results; explain why this is not an equal-total-budget experiment.

The starter runs only one additional case, so Run All remains inexpensive. Extend the list yourself. Do not simulate the held-out test set.

In [ ]:
homework_budgets = [20]  # Extend to [20, 40, 60] for the assignment.
homework_rows = []
for m in homework_budgets:
    for method in ["random", "kmeans"]:
        result = run_compressor_experiment(method, m=m, seed=COMPRESSOR_SEED)
        homework_rows.append(result["row"])
homework_budget_table = pd.DataFrame(homework_rows)
display(homework_budget_table[[
    "compressor", "references", "reference_payoff_paths",
    "validation_mean_radius", "validation_RMSE_pp"
]].round(5))

## Homework 2 — A compressor designed for coverage

**Core; about 45–60 minutes.** A farthest-first selector starts at one candidate and repeatedly adds the candidate farthest from its nearest selected reference. The starter provides a complete small implementation.

Use its selected references in the same MC-and-copy pipeline. Compare its mean radius, 95th-percentile radius, valuation RMSE, and selection time with the two guided compressors at the same reference and path counts. Does prioritizing geometric coverage necessarily improve pricing? Give evidence, without claiming that one seed establishes a general rule.

In [ ]:
def farthest_first_positions(Z, m, seed=2027):
    """Greedy diversity heuristic using features only; not an optimal solution."""
    Z = np.asarray(Z)
    rng = np.random.default_rng(seed)
    first = int(rng.integers(len(Z)))
    chosen = [first]
    available = np.ones(len(Z), dtype=bool)
    available[first] = False
    minimum_d2 = np.sum((Z - Z[first])**2, axis=1)
    while len(chosen) < m:
        eligible = np.where(available, minimum_d2, -np.inf)
        nxt = int(np.argmax(eligible))
        chosen.append(nxt)
        available[nxt] = False
        minimum_d2 = np.minimum(
            minimum_d2, np.sum((Z - Z[nxt])**2, axis=1)
        )
    return np.sort(np.asarray(chosen, dtype=int))

t0 = perf_counter()
diverse_positions = farthest_first_positions(
    Zcandidate, N_REFERENCES, seed=COMPRESSOR_SEED
)
diversity_selection_seconds = perf_counter() - t0
diverse_radii = coverage_radii(Zvalidation, Zcandidate[diverse_positions])
display(pd.DataFrame([{
    "references": len(diverse_positions),
    "selection_seconds": diversity_selection_seconds,
    "validation_mean_radius": diverse_radii.mean(),
    "validation_p95_radius": np.quantile(diverse_radii, 0.95)
}]))
# Continue by simulating candidate.iloc[diverse_positions] with the same
# reference seed/path count and then calling nearest_predict and score_predictions.

## Homework 3 — Diagnose a counterexample (optional)

Choose three validation contracts where the methods disagree most. Inspect their original features, selected reference features, MC standard errors, and errors. Write 150–200 words explaining one failure case. A case study suggests a hypothesis; it does not estimate an overall effect.

The table below is a starting point. Its response differences use validation data and are for development only.

In [ ]:
case_table = validation[["contract_id"] + FEATURES].copy()
case_table["validation_MC"] = y_validation
case_table["validation_SE"] = validation_mc["se"].to_numpy()
case_table["random_prediction"] = runs["random"]["prediction"]
case_table["kmeans_prediction"] = runs["kmeans"]["prediction"]
case_table["prediction_disagreement_pp"] = 100 * np.abs(
    case_table["random_prediction"] - case_table["kmeans_prediction"]
)
display(case_table.nlargest(3, "prediction_disagreement_pp"))

## Weekly deliverable checklist

- [ ] I can identify the compressor, simulator, and predictor in the code.
- [ ] My selected references are distinct candidate contracts.
- [ ] My scaler was fitted only on candidate features; premium and ID are excluded.
- [ ] My comparison uses the same path count, reference count, and validation label bank.
- [ ] I report geometric coverage separately from validation pricing error.
- [ ] My currency predictions multiply by the query's own premium.
- [ ] My ledger includes validation work and explains whether costs are shared.
- [ ] I answered the reading questions and completed the two core coding exercises.
- [ ] My interpretation quotes observed results and states one limitation.
- [ ] I have not accessed test data or used exact prices as training/tuning labels.

**Bring to the meeting:** the executed comparison table, the two main figures, a five-sentence research-log entry, and one question you would like Week 5's smoother predictors to address.